In [1]:
import numpy as np
import time
from mcts.decoupled_mcts import MCTS
from simulator.mcts_game_state import MCTSGameState
from simulator.mcts_game_state import navigation_rollout

In [2]:
from mcts.decoupled_mcts import MCTSConfig
from simulator.mcts_game_state import MCTSGameStateConfig
from simulator.agents import Robot
from simulator.map import ScenarioMap


num_humans = 5
num_actions = 3
tree_depth = 6
dt = 0.5
human_speed = 1.7

manual_goal = (16, 5)

scenario = ScenarioMap.build_default()
start_position = scenario.cell_to_world((0, 4))

robot = Robot(
    position=start_position,
    theta=0.0,
)

human_positions = np.zeros((num_humans, 2)) + start_position
human_velocities = np.zeros((num_humans, 2))
human_velocities[:, 0] = 1.0
human_orientations = np.arctan2(human_velocities[:, 1], human_velocities[:, 0])
human_goals = human_positions + human_velocities * (human_speed * dt * tree_depth)

positions = np.vstack((robot.position[None, :], human_positions))
orientations = np.concatenate((np.array([robot.theta], dtype=np.float32), human_orientations))
robot_goal = scenario.cell_to_world(manual_goal)[None, :]
goal_positions = np.vstack((robot_goal, human_goals))

num_agents = positions.shape[0]

mcts_config = MCTSConfig(
    num_actors=num_agents,
    num_actions=num_actions,
    max_depth=6
)
state_config = MCTSGameStateConfig(
    mcts_config=mcts_config,
    movement_distance=human_speed * dt,
    angle=np.pi / 4.0,
    uncomfortable_distance=1.0,
    map=scenario,
)

mcts = MCTS(mcts_config, navigation_rollout)

root_state = MCTSGameState(
    positions=positions,
    orientations=orientations,
    agent_goal_positions=goal_positions,
    value_accumulator=None,
    config=state_config,
    depth=0
)

In [3]:
timings = []
for i in range(10):
    start_time = time.perf_counter()
    _, child_state, stats = mcts.search(root_state, num_simulations=500)
    end_time = time.perf_counter()
    timings.append((end_time - start_time) * 1000)

print(timings)

[176.83807900175452, 138.21930496487767, 137.7811130369082, 136.11579296411946, 135.93181100441143, 141.58179599326104, 177.8694349923171, 140.38584800437093, 145.354459004011, 141.41014503547922]


In [ ]:
_, child_state, stats = mcts.search(root_state, num_simulations=500)

In [ ]:
stats

In [ ]:
import math


def _select_child(config, node):
    sqrt_visits = math.sqrt(node.visits + 1)

    actions = [0] * config.num_actors

    for actor in range(config.num_actors):
        best_score = -math.inf
        best_action = 0
        for action in range(config.num_actions):
            action_visits = node.visits_by_action[actor][action]
            q = node.value_by_action[actor][action] / action_visits if action_visits > 0 else 0.0
            u = 2.0 * (sqrt_visits / (1 + action_visits))
            score = q + u
            if score > best_score:
                best_score = score
                best_action = action
        actions[actor] = best_action

    return actions

In [ ]:
visits = np.random.randint(1, 10, (num_agents * num_actions))
values = np.random.random((num_agents * num_actions))
zeros = np.zeros_like(values)

def _fast_select_child(node):
    sqrt_visits = math.sqrt(node.visits + 1)

    q = np.divide(values, visits, out=zeros, where=visits!=0)
    u = 2.0 * (sqrt_visits / (1 + visits))
    scores = q + u
    split_scores = np.split(scores, num_agents)
    actions = np.argmax(split_scores, axis=1)
    return actions

In [ ]:
_fast_select_child(node)

In [ ]:
from mcts.decoupled_mcts import _Node


node = _Node(
    root_state,
    mcts_config,
    None,
    None
)

timings = []
for i in range(10):
    start_time = time.perf_counter()
    #_select_child(mcts_config, node)
    _fast_select_child(node)
    end_time = time.perf_counter()
    timings.append((end_time - start_time) * 1000)

print(timings)